# FormuLab CV: Synthetic Visual Screening

Kaggle-ready pipeline untuk melatih baseline classifier pada dataset procedural synthetic FormuLab v3 yang memakai domain randomization. Dataset ini hanya mendemonstrasikan workflow CV dan **bukan** validasi stabilitas kosmetik nyata. Jalankan notebook setelah mengunggah folder `cv/data/synthetic_v3/` sebagai Kaggle Dataset.

## Runbook Kaggle

1. Buat Kaggle Dataset dari folder `cv/data/synthetic_v3/`.
2. Attach dataset itu ke notebook Kaggle.
3. Set `DATASET_ROOT` di cell konfigurasi ke lokasi input yang muncul di `/kaggle/input/`.
4. Aktifkan GPU bila tersedia, lalu Run All.
5. Unduh output dari `/kaggle/working/formulab_cv_outputs/`.

Pipeline memvalidasi manifest, memisahkan train/validation/test menurut manifest, memilih epoch dari validation loss, lalu menyentuh test set sekali pada evaluasi akhir.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 2026
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 70
LEARNING_RATE = 1e-3
NUM_WORKERS = 2

# Update this path to match the Kaggle input slug after attaching the dataset.
DATASET_ROOT = Path("/kaggle/input/formulab-cv-synthetic/synthetic_v3")
OUTPUT_DIR = Path("/kaggle/working/formulab_cv_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"device": str(device), "dataset_root": str(DATASET_ROOT), "output_dir": str(OUTPUT_DIR)})
assert DATASET_ROOT.is_dir(), f"Dataset tidak ditemukan: {DATASET_ROOT}. Edit DATASET_ROOT sesuai /kaggle/input/."

In [ ]:
LABELS = ("stable_uniform", "creaming", "phase_separation", "heterogeneous")
SPLITS = ("train", "validation", "test")

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def load_manifest(dataset_root):
    path = dataset_root / "manifest.jsonl"
    assert path.is_file(), f"Manifest tidak ditemukan: {path}"
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    assert rows, "Manifest kosong"
    return rows

def validate_manifest(dataset_root, rows):
    errors = []
    sequence_splits = {}
    class_counts = defaultdict(Counter)
    for row in rows:
        image_id = row.get("image_id", "<unknown>")
        if row.get("label") not in LABELS:
            errors.append(f"{image_id}: unknown label")
        if row.get("split") not in SPLITS:
            errors.append(f"{image_id}: unknown split")
        if row.get("data_origin") != "synthetic_demo":
            errors.append(f"{image_id}: non-synthetic provenance")
        if row.get("scientific_validation_status") != "not_validated_for_production":
            errors.append(f"{image_id}: invalid scientific status")
        image_path = dataset_root / row["image_path"]
        if not image_path.is_file():
            errors.append(f"{image_id}: missing image")
        elif sha256(image_path) != row["image_sha256"]:
            errors.append(f"{image_id}: hash mismatch")
        sequence_id = row["sample_sequence_id"]
        prior_split = sequence_splits.setdefault(sequence_id, row["split"])
        if prior_split != row["split"]:
            errors.append(f"{image_id}: sequence leaks across splits")
        class_counts[row["split"]][row["label"]] += 1
    for split in SPLITS:
        missing = set(LABELS) - set(class_counts[split])
        if missing:
            errors.append(f"{split}: missing labels {sorted(missing)}")
    return errors, class_counts

rows = load_manifest(DATASET_ROOT)
errors, class_counts = validate_manifest(DATASET_ROOT, rows)
assert not errors, "Manifest invalid:\n" + "\n".join(errors[:20])

manifest = pd.DataFrame(rows)
label_to_index = {label: index for index, label in enumerate(LABELS)}
index_to_label = {index: label for label, index in label_to_index.items()}
print(manifest.groupby(["split", "label"]).size().unstack(fill_value=0))
print(f"Validated {len(manifest)} images across {manifest.sample_sequence_id.nunique()} sequences.")

In [ ]:
fig, axes = plt.subplots(1, len(LABELS), figsize=(18, 4))
for axis, label in zip(axes, LABELS):
    row = manifest[manifest.label.eq(label)].iloc[0]
    axis.imshow(Image.open(DATASET_ROOT / row.image_path).convert("RGB"))
    axis.set_title(label)
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.96, 1.04)),
    transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

class ManifestImageDataset(Dataset):
    def __init__(self, frame, dataset_root, transform):
        self.frame = frame.reset_index(drop=True)
        self.dataset_root = Path(dataset_root)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(self.dataset_root / row.image_path).convert("RGB")
        return self.transform(image), label_to_index[row.label], row.image_id

def make_loader(split, transform, shuffle):
    subset = manifest[manifest.split.eq(split)]
    dataset = ManifestImageDataset(subset, DATASET_ROOT, transform)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda"), dataset

train_loader, train_dataset = make_loader("train", train_transform, True)
validation_loader, validation_dataset = make_loader("validation", eval_transform, False)
test_loader, test_dataset = make_loader("test", eval_transform, False)
print({"train": len(train_dataset), "validation": len(validation_dataset), "test": len(test_dataset)})

In [ ]:
class TinyVialCNN(nn.Module):
    """Small offline baseline suited to the limited synthetic-demo corpus."""
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 192, kernel_size=3, padding=1), nn.BatchNorm2d(192), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.25), nn.Linear(192 * 4 * 4, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

model = TinyVialCNN(num_classes=len(LABELS)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
print(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad), "trainable parameters")

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_loss, correct, count = 0.0, 0, 0
    for images, targets, _ in loader:
        images, targets = images.to(device), targets.to(device)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, targets)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * targets.size(0)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        count += targets.size(0)
    return {"loss": total_loss / count, "accuracy": correct / count}

history = []
best_validation_loss = float("inf")
best_epoch = None
best_model_path = OUTPUT_DIR / "tiny_vial_cnn_best.pt"
patience, stale_epochs = 15, 0
for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = run_epoch(train_loader, training=True)
    validation_metrics = run_epoch(validation_loader, training=False)
    history.append({"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"validation_{k}": v for k, v in validation_metrics.items()}})
    print(f"epoch={epoch:02d} train_loss={train_metrics['loss']:.4f} val_loss={validation_metrics['loss']:.4f} val_acc={validation_metrics['accuracy']:.3f}")
    if validation_metrics["loss"] < best_validation_loss:
        best_validation_loss = validation_metrics["loss"]
        best_epoch = epoch
        stale_epochs = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "labels": LABELS,
            "image_size": IMAGE_SIZE,
            "dataset_version": manifest.generator_version.iloc[0],
            "data_origin": "synthetic_demo",
            "scientific_validation_status": "not_validated_for_production",
            "best_epoch": best_epoch,
        }, best_model_path)
    else:
        stale_epochs += 1
        if stale_epochs >= patience:
            print(f"Early stopping at epoch {epoch}; best epoch={best_epoch}.")
            break

history_frame = pd.DataFrame(history)
history_frame.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print({"best_epoch": best_epoch, "best_validation_loss": best_validation_loss})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_frame.epoch, history_frame.train_loss, label="train")
axes[0].plot(history_frame.epoch, history_frame.validation_loss, label="validation")
axes[0].set(title="Loss", xlabel="Epoch")
axes[0].legend()
axes[1].plot(history_frame.epoch, history_frame.train_accuracy, label="train")
axes[1].plot(history_frame.epoch, history_frame.validation_accuracy, label="validation")
axes[1].set(title="Accuracy", xlabel="Epoch", ylim=(0, 1))
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
@torch.inference_mode()
def predict(loader):
    model.eval()
    targets, predictions, probabilities, image_ids = [], [], [], []
    for images, batch_targets, batch_ids in loader:
        logits = model(images.to(device))
        probs = logits.softmax(dim=1).cpu().numpy()
        targets.extend(batch_targets.numpy().tolist())
        predictions.extend(probs.argmax(axis=1).tolist())
        probabilities.extend(probs.tolist())
        image_ids.extend(batch_ids)
    return np.array(targets), np.array(predictions), np.array(probabilities), image_ids

test_targets, test_predictions, test_probabilities, test_image_ids = predict(test_loader)
test_accuracy = float((test_targets == test_predictions).mean())
print({"synthetic_demo_test_accuracy": test_accuracy, "test_examples": len(test_targets)})
print(classification_report(test_targets, test_predictions, target_names=LABELS, zero_division=0))

figure, axis = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix(test_targets, test_predictions, labels=range(len(LABELS))), display_labels=LABELS).plot(ax=axis, xticks_rotation=35, colorbar=False)
axis.set_title("Synthetic-demo test confusion matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "test_confusion_matrix.png", dpi=160)
plt.show()

In [ ]:
test_predictions_frame = pd.DataFrame({
    "image_id": test_image_ids,
    "target": [index_to_label[value] for value in test_targets],
    "prediction": [index_to_label[value] for value in test_predictions],
    "confidence": test_probabilities.max(axis=1),
})
test_predictions_frame.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
report = {
    "model_name": "TinyVialCNN",
    "dataset_version": manifest.generator_version.iloc[0],
    "data_origin": "synthetic_demo",
    "scientific_validation_status": "not_validated_for_production",
    "split_unit": "sample_sequence_id; stratified by visual label",
    "labels": list(LABELS),
    "best_epoch": best_epoch,
    "test_accuracy": test_accuracy,
    "limitations": [
        "Model dilatih hanya pada render procedural synthetic FormuLab.",
        "Metric ini tidak membuktikan performa pada foto sampel kosmetik nyata.",
        "Output hanya synthetic-demo visual screening dan tidak menggantikan stability test atau review formulator.",
    ],
}
(OUTPUT_DIR / "training_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print(f"Saved Kaggle artifacts to: {OUTPUT_DIR}")